# Radar through the winter — Baltic sea ice with Sentinel-1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/09_sar_baltic_ice.ipynb)

Every animation so far used **optical** sensors, and all of them share one
weakness: clouds. A Baltic winter is their worst case — weeks of overcast and
eight hours of grey daylight. **Synthetic-aperture radar (SAR)** does not care:
it brings its own illumination and its microwaves pass through clouds, so
Sentinel-1 images the sea ice in the Bodden lagoons on every single pass, day
and night, December through April.

This notebook animates the **winter 2025/26** over the Rügen–Usedom coast. The
radar never joined the package's sensor registry (its physics fits none of the
optical conventions — no clouds to mask, backscatter instead of reflectance),
so the Earth Engine work here is raw and explicit; from the package we take
auth, AOI handling and the frame annotation.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code

## 2 · Authenticate

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The scene and the season

The AOI spans the shallow lagoons between Rügen and Usedom — the Greifswalder
Bodden freezes in most cold spells while the open Baltic beside it stays liquid,
which is exactly the contrast radar shows well.

In [ ]:
from pathlib import Path
import ee
from gee_animation import aoi

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "balticsea_ice" / "aoi.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = (f"https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/{BRANCH}"
           "/docs/aoi/balticsea_ice/aoi.geojson")
    urllib.request.urlretrieve(RAW, "aoi.geojson")
    REGION = "aoi.geojson"

START, END = "2025-12-01", "2026-05-01"      # the ice season, end exclusive

region_geom = aoi.parse({"geojson": REGION})
bbox = region_geom.bounds().coordinates().get(0).getInfo()
print("AOI corners:", [[round(c, 2) for c in p] for p in bbox[:-1]])

## 4 · Ask the archive before assuming anything

Sentinel-1 scenes differ by **instrument mode** (IW over most land and coasts,
EW in the Arctic) and **polarisation** (VV, VH, HH …). Which combination
actually covers *this* AOI *this* winter is a question for the archive, not for
a tutorial's assumption:

In [ ]:
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(region_geom)
      .filterDate(START, END))

combos = s1.aggregate_histogram(
    "instrumentMode").getInfo()
pols = s1.aggregate_histogram("transmitterReceiverPolarisation").getInfo()
print("scenes by mode:        ", combos)
print("scenes by polarisation:", pols)

BAND = "VV"          # chosen from the inventory above: IW VV+VH dominates here
s1 = (s1.filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", BAND))
        .select(BAND))
print(f"usable {BAND} scenes:", s1.size().getInfo())

## 5 · Ten-day backscatter mosaics

Sea ice moves and grows on a scale of days, so monthly frames would blur the
story. Ten-day medians keep the motion and still smooth the speckle a little.
`S1_GRD` delivers backscatter in **dB** (a log scale); the median of dB values
is fine for a picture — just remember it is not the mean of the physical power.

Reading the frames: wind-roughened **open water scatters strongly** (bright);
**smooth new ice is dark**; ridged or snow-covered ice sits in between. Land is
bright year-round.

In [ ]:
import io
import urllib.request
import numpy as np
from datetime import date, timedelta
from PIL import Image as PILImage
from gee_animation.render import annotate

VIZ = {"min": -25, "max": 0}                 # dB, fixed across all frames
STEP = 10                                     # days per frame

def periods(start, end, step):
    d0, d1 = date.fromisoformat(start), date.fromisoformat(end)
    while d0 < d1:
        dn = min(d0 + timedelta(days=step), d1)
        if (dn - d0).days >= step // 2:      # drop a trailing stub period
            yield str(d0), str(dn)
        d0 = dn

frames, labels = [], []
for p0, p1 in periods(START, END, STEP):
    chunk = s1.filterDate(p0, p1)
    n = chunk.size().getInfo()
    if n == 0:
        print(f"{p0} … skipped (no scenes)")
        continue
    mosaic = chunk.median().clip(region_geom)
    url = mosaic.getThumbURL({**VIZ, "region": region_geom, "dimensions": 900,
                              "format": "png"})
    png = PILImage.open(io.BytesIO(urllib.request.urlopen(url).read())).convert("RGBA")
    canvas = PILImage.new("RGBA", png.size, (40, 40, 40, 255))   # no-data -> dark grey
    canvas.alpha_composite(png)
    rgb = np.array(canvas.convert("RGB"))
    frames.append(annotate(rgb, f"{p0} · {n} passes", credit="Sentinel-1 VV, contains modified Copernicus data"))
    print(f"{p0} … {n} scenes")
print(len(frames), "frames")

## 6 · Assemble and watch

In [ ]:
import imageio.v2 as imageio

# Crop to multiples of 16 so the H.264 encoder needs no resampling.
h16, w16 = (frames[0].shape[0] // 16) * 16, (frames[0].shape[1] // 16) * 16
frames = [f[:h16, :w16] for f in frames]

out = Path("out"); out.mkdir(exist_ok=True)
gif = out / "baltic_ice_2025_26.gif"
imageio.mimwrite(gif, frames, duration=1000 / 2, loop=0)        # 2 frames/s
try:
    mp4 = out / "baltic_ice_2025_26.mp4"
    imageio.mimwrite(mp4, frames, fps=2, codec="libx264", quality=8)
    print("wrote", gif, "and", mp4)
except Exception as e:                       # ffmpeg missing -> GIF still stands
    mp4 = None
    print("wrote", gif, f"(mp4 skipped: {e})")

from IPython.display import Image as IPyImage, display
display(IPyImage(filename=str(gif)))

## 7 · What optical would have given you

The same AOI, the same winter, Sentinel-2 with the series' usual 70 % scene
cloud filter — count the usable scenes per month and the case for radar makes
itself:

In [ ]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(region_geom).filterDate(START, END)
      .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", 70)))
per_month = s2.aggregate_histogram("system:index").getInfo()
counts = {}
for k in per_month:
    counts[k[:6]] = counts.get(k[:6], 0) + 1
for m in sorted(counts):
    print(f"{m[:4]}-{m[4:6]}: {counts[m]:>3} optical scenes under 70% cloud")
print("radar passes in the same window:", s1.size().getInfo())

## Make it yours

- **VH instead of VV** (`BAND = "VH"`, viz max ≈ −5): cross-polarisation darkens
  open water further and makes deformed ice stand out.
- **A real pack-ice winter**: move the AOI to the Bothnian Bay (around
  `[20.5, 64.5, 25.5, 65.9]`) where the Baltic freezes every year, shift the viz,
  and re-run — the archive inventory cell will tell you if EW mode takes over.
- **Another season**: the 2024/25 or 2026/27 winter is one date change away.
- **Sharper time**: `STEP = 6` matches Sentinel-1's repeat cycle; frames get
  noisier but the ice edge moves almost scene by scene.
- Compare your animation to the
  [FMI ice charts](https://en.ilmatieteenlaitos.fi/ice-conditions) for the same
  weeks — the professional ground truth for this sea.